# Stage 7 — MAUDE clinical anchor

Link devices to FDA's adverse-event database (MAUDE) two ways: family-level event counts per
product code, and a device-level comparison of recalled predicates vs. a matched random sample of
non-recalled predicates. The stable, drift-resistant finding is the **ratio**, not the absolute
counts (MAUDE grows continuously).

Network: yes — MAUDE is large and not part of the frozen snapshot by default; these are live counts
reported "as of" the run date. Uses a fixed seed (42) for the matched sample.

In [ ]:
import urllib.request, urllib.parse, json, time, random
import pandas as pd
from scipy.stats import mannwhitneyu

corpus = pd.read_csv("data/corpus.csv", dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
edges = pd.read_csv("data/predicate_edges.csv", dtype=str)
recalled = pd.read_csv("data/recalled_nodes.csv", dtype=str)
manifest = json.load(open("snapshot/SNAPSHOT.json"))
CODES = manifest["product_codes"]

import urllib.error, os
OPENFDA_KEY = os.environ.get("OPENFDA_KEY", "").strip()   # optional free key -> 120k/day
_last_req = [0.0]
def event_count(params, _tries=20):
    if OPENFDA_KEY:
        params = {**params, "api_key": OPENFDA_KEY}
    url = "https://api.fda.gov/device/event.json?" + urllib.parse.urlencode(params)
    delay = 1.0
    for attempt in range(1, _tries + 1):
        wait = _last_req[0] + 0.40 - time.time()   # ~150/min unauthenticated
        if wait > 0:
            time.sleep(wait)
        _last_req[0] = time.time()
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return json.loads(r.read())
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return {"meta": {"results": {"total": 0}}, "results": []}
            if e.code in (429, 500, 502, 503, 504) and attempt < _tries:
                time.sleep(delay); delay = min(delay * 2, 60); continue
            raise
        except (urllib.error.URLError, TimeoutError):
            if attempt < _tries:
                time.sleep(delay); delay = min(delay * 2, 60); continue
            raise


### Family-level event totals

In [ ]:
fam = []
for c in CODES:
    j = event_count({"search": f"device.device_report_product_code:{c}", "limit": 1})
    tot = j["meta"]["results"]["total"]
    if tot == 0:
        continue
    d = {"product_code": c, "total": tot}
    jc = event_count({"search": f"device.device_report_product_code:{c}", "count": "event_type.exact"})
    for row in jc.get("results", []):
        d[row["term"].lower()] = row["count"]
    fam.append(d)
    time.sleep(0.05)
famdf = pd.DataFrame(fam).fillna(0)
famdf.to_csv("data/maude_family.csv", index=False)
tot_events = int(famdf["total"].sum())
tot_death = int(famdf.get("death", pd.Series([0])).sum())
tot_injury = int(famdf.get("injury", pd.Series([0])).sum())
print(f"CHECKPOINT  MAUDE family totals: events {tot_events} | deaths {tot_death} | injuries {tot_injury}")

### Device-level comparison: recalled predicates vs matched non-recalled predicates

In [ ]:
cited_preds = set(edges["predicate_knumber"])
recalled_set = set(recalled["k_number"])
recalled_preds = sorted(cited_preds & recalled_set)
nonrecalled_preds = sorted(cited_preds - recalled_set)

random.seed(42)
sample_nonrec = random.sample(nonrecalled_preds, min(len(recalled_preds), len(nonrecalled_preds)))

def dev_events(k):
    j = event_count({"search": f'pma_pmn_number:"{k}"', "limit": 1})
    return j["meta"]["results"]["total"]

def collect(klist):
    rows = []
    for k in klist:
        n = dev_events(k)
        yr = corpus.loc[corpus["k_number"] == k, "decision_year"]
        yr = yr.iloc[0] if len(yr) else None
        age = (2026 - yr) if pd.notna(yr) else None
        rows.append({"k_number": k, "events": n, "year": yr,
                     "events_per_year": (n / age) if age and age > 0 else 0})
        time.sleep(0.03)
    return pd.DataFrame(rows)

rec_df = collect(recalled_preds)
non_df = collect(sample_nonrec)
rec_df.to_csv("data/maude_recalled.csv", index=False)
non_df.to_csv("data/maude_nonrecalled.csv", index=False)

rec_match = (rec_df["events"] > 0).mean()
non_match = (non_df["events"] > 0).mean()
u, p = mannwhitneyu(rec_df["events_per_year"], non_df["events_per_year"], alternative="greater")
print(f"CHECKPOINT  match rate {rec_match*100:.0f}% vs {non_match*100:.0f}%")
print(f"CHECKPOINT  median events {rec_df['events'].median():.0f} vs {non_df['events'].median():.0f}")
print(f"CHECKPOINT  events/yr {rec_df['events_per_year'].median():.2f} vs "
      f"{non_df['events_per_year'].median():.2f} | Mann-Whitney p={p:.2e}")

json.dump({"family_events": tot_events, "family_deaths": tot_death,
           "recalled_match_rate": round(rec_match, 3), "nonrecalled_match_rate": round(non_match, 3),
           "mw_p": p}, open("data/maude_comparison.json", "w"), indent=2)